---

# HoloMine Property Price Prediction

*Real-estate-adapted DeBERTa-v3 with numeric feature extraction, TF-IDF, and late-fusion stacking.*

---

## HoloMine Task 2

- Competition pipeline: DeBERTa-v3 domain adaptation and regression
- Complementary branches: word/character TF-IDF and structured text extraction
- Final prediction: cross-validated late fusion / stacking

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Data Preparation and Feature Extraction**](#3)
4. [**Domain-Adaptive Pretraining**](#4)
5. [**DeBERTa Regression Fine-Tuning**](#5)
6. [**TF-IDF + Ridge Branch**](#6)
7. [**Structured CatBoost Branch**](#7)
8. [**Late Fusion / Stacking**](#8)
9. [**Diagnostics**](#9)
10. [**Submission**](#10)

---

# Introduction <a name="1"></a>

---

## Overview

*HoloMine Task 2 asks a regression model to infer property `listPrice` from listing prose alone. The text mixes semantic signals such as location and property quality with explicit numeric signals such as bedrooms, bathrooms, area, fees, dates, and occasionally monetary amounts.*

## Aim

*This notebook implements the full NB-1 pipeline: continued masked-language-model pretraining of DeBERTa-v3 on real-estate descriptions, cross-validated regression on `log1p(listPrice)`, two complementary classical branches, and an honest second-level stack. The default configuration is designed for a single RTX 5090 on Vast.ai while keeping every major component switchable and resumable.*

## Metric

***Mean Absolute Error (MAE)** is the competition metric. It measures the average absolute price error in the original price scale, so improvements on expensive tail listings can materially affect the score.*

$$\mathrm{MAE}=\frac{1}{n}\sum_{i=1}^{n}|y_i-\hat{y}_i|$$

## Dataset

*The supplied competition files contain 14,640 labelled training listings and 3,659 test listings. Each row has a unique `id` and one free-text property description; `listPrice` is present only in the training split.*

*All structured features used below are extracted from the provided text rather than sourced externally.*

```
data/
|-- train.csv              # id, text, listPrice
|-- test.csv               # id, text
`-- sample_submission.csv  # id, listPrice
```

## Approach: RealEstate DeBERTa + Late Fusion

*The semantic branch starts from `microsoft/deberta-v3-base`. A `<NUM>` marker is inserted before numeric expressions, then the model is further pretrained with masked language modelling on the listing corpus. The adapted encoder is fine-tuned fold-by-fold to predict `log1p(listPrice)`.*

*Two independent branches preserve complementary information: word/character TF-IDF + Ridge captures rare lexical and location tokens, while CatBoost receives regex-derived quantities and property cues. Their out-of-fold predictions are combined by a second-level model, preventing the stacker from training on in-sample base predictions.*

```
listing text
    |
    +--> numeric-aware text --> RealEstate DeBERTa --> OOF log-price --+
    |                                                                  |
    +--> word + char TF-IDF --> Ridge -----------------> OOF log-price +--> stacker --> listPrice
    |                                                                  |
    `--> regex / keyword features --> CatBoost --------> OOF log-price -+
```

*Five folds are stratified by ranked log-price quantiles. The notebook reports overall MAE, log-MAE, and top-decile MAE, then trains the final stacker on all OOF meta-features.*

---

# Initialization <a name="2"></a>

---

## Environment Setup

The notebook assumes a recent CUDA-enabled PyTorch image on Vast.ai. PyTorch itself is intentionally not reinstalled so that the CUDA build supplied by the image remains intact; the remaining libraries are pinned to compatible major versions.

In [1]:
!nvidia-smi

Sun Sep 20 09:46:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        On  |   00000000:01:00.0 Off |                  N/A |
|  0%   34C    P5             19W /  500W |       2MiB /  32607MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

The following cell installs all non-PyTorch libraries used in this notebook.

In [2]:
%pip install -q "transformers>=4.51,<5" "datasets>=3.5,<4" "accelerate>=1.6,<2" "sentencepiece>=0.2" "protobuf>=5.26" "catboost>=1.2.8" "scikit-learn>=1.6,<1.9" "scipy>=1.13" "pandas>=2.2" "matplotlib>=3.9" "joblib>=1.4"

Note: you may need to restart the kernel to use updated packages.


## Import Libraries

All imports are centralised here so later cells only contain modelling logic.

In [28]:
import gc
import html
import inspect
import json
import math
import os
import random
import re
import shutil
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
import torch
import transformers
from catboost import CatBoostRegressor
from datasets import Dataset, concatenate_datasets
from scipy.sparse import hstack
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import (
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

## Seed Everything

A single seed is propagated through Python, NumPy, PyTorch, scikit-learn, Transformers, and CatBoost-facing configuration.

In [4]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

seed_everything(42)

## Settings

All paths and training hyperparameters are centralised below. `HOLOMINE_DATA_DIR` can be exported before launching Jupyter; otherwise the notebook detects a nearby directory containing `train.csv` and `test.csv`. Expensive branches can be disabled when reusing saved predictions.

In [5]:
class Settings:
    SEED = 42
    RUN_NAME = 'nb1_deberta_late_fusion'
    BASE_MODEL = 'microsoft/deberta-v3-base'
    NUM_TOKEN = '<NUM>'
    ADD_NUM_TOKEN = True

    N_FOLDS = 5
    STRAT_BINS = 20

    RUN_DAPT = True
    FORCE_DAPT = False
    DAPT_USE_TEST_TEXT = True
    DAPT_EPOCHS = 3.0
    DAPT_LR = 5e-5
    DAPT_BATCH_SIZE = 16
    DAPT_GRAD_ACCUM = 4
    DAPT_MLM_PROBABILITY = 0.15
    DAPT_MAX_LENGTH = 512

    RUN_DEBERTA = True
    FOLDS_TO_RUN = tuple(range(N_FOLDS))
    FT_EPOCHS = 4.0
    FT_LR = 2e-5
    FT_BATCH_SIZE = 16
    FT_EVAL_BATCH_SIZE = 32
    FT_GRAD_ACCUM = 4
    FT_WEIGHT_DECAY = 0.01
    FT_WARMUP_RATIO = 0.08
    FT_MAX_LENGTH = 512
    FT_EARLY_STOPPING_PATIENCE = 1
    GRADIENT_CHECKPOINTING = True

    RUN_TFIDF = True
    TFIDF_WORD_MAX_FEATURES = 150_000
    TFIDF_CHAR_MAX_FEATURES = 180_000
    TFIDF_WORD_MIN_DF = 2
    TFIDF_CHAR_MIN_DF = 3
    TFIDF_RIDGE_ALPHA = 12.0
    SAVE_CLASSICAL_FOLDS = True

    RUN_STRUCTURED = True
    CATBOOST_TASK_TYPE = 'CPU'
    CATBOOST_ITERATIONS = 3000
    CATBOOST_DEPTH = 7
    CATBOOST_LR = 0.03
    CATBOOST_L2 = 6.0
    CATBOOST_EARLY_STOP = 200

    RUN_STACKING = True
    META_RIDGE_ALPHA = 8.0
    META_CATBOOST_ITERATIONS = 1200
    META_CATBOOST_DEPTH = 4
    META_CATBOOST_LR = 0.025
    META_EARLY_STOP = 120

    RUN_KNN_BASELINE = False
    KNN_K = 5

    MAX_PRICE_PRED = 200_000_000.0
    NUM_WORKERS = min(8, os.cpu_count() or 2)

    @staticmethod
    def detect_data_dir():
        env_value = os.getenv('HOLOMINE_DATA_DIR', '').strip()
        candidates = []
        if env_value:
            candidates.append(Path(env_value).expanduser())

        cwd = Path.cwd()
        candidates.extend([
            cwd / 'data',
            cwd,
            cwd / 'hology-2' / 'data',
            cwd.parent / 'data',
            cwd.parent / 'hology-2' / 'data',
            Path('/workspace/data'),
            Path('/workspace/hology-2/data'),
            Path('/root/data'),
        ])

        kaggle_root = Path('/kaggle/input')
        if kaggle_root.exists():
            candidates.extend(sorted(kaggle_root.glob('*')))
            candidates.extend(sorted(kaggle_root.glob('*/*')))

        seen = set()
        for candidate in candidates:
            candidate = candidate.resolve() if candidate.exists() else candidate
            key = str(candidate)
            if key in seen:
                continue
            seen.add(key)
            if (candidate / 'train.csv').exists() and (candidate / 'test.csv').exists():
                return candidate

        raise FileNotFoundError(
            'Could not locate train.csv and test.csv. Set HOLOMINE_DATA_DIR=/path/to/data.'
        )

    DATA_DIR = None
    OUTPUT_DIR = None
    ARTIFACT_DIR = None
    FIG_DIR = None
    DAPT_DIR = None
    FOLD_MODEL_DIR = None

CFG = Settings()
CFG.DATA_DIR = CFG.detect_data_dir()
_on_kaggle = Path('/kaggle/working').exists()
CFG.OUTPUT_DIR = (Path('/kaggle/working') if _on_kaggle else Path.cwd() / 'outputs') / CFG.RUN_NAME
CFG.ARTIFACT_DIR = CFG.OUTPUT_DIR / 'artifacts'
CFG.FIG_DIR = CFG.OUTPUT_DIR / 'figures'
CFG.DAPT_DIR = CFG.ARTIFACT_DIR / 'realestate_deberta_mlm'
CFG.FOLD_MODEL_DIR = CFG.ARTIFACT_DIR / 'deberta_folds'

for path in [CFG.OUTPUT_DIR, CFG.ARTIFACT_DIR, CFG.FIG_DIR, CFG.FOLD_MODEL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HOME'] = str(CFG.OUTPUT_DIR / 'hf_cache')

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

USE_BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
USE_FP16 = bool(torch.cuda.is_available() and not USE_BF16)

print('DATA_DIR:', CFG.DATA_DIR)
print('OUTPUT_DIR:', CFG.OUTPUT_DIR)
print('CUDA:', torch.cuda.is_available(), '| BF16:', USE_BF16, '| FP16:', USE_FP16)
print('Transformers:', transformers.__version__, '| Torch:', torch.__version__)

DATA_DIR: /workspace/data
OUTPUT_DIR: /workspace/outputs/nb1_deberta_late_fusion
CUDA: True | BF16: True | FP16: False
Transformers: 4.56.2 | Torch: 2.11.0+cu128


## Load Dataset

The competition files are loaded without visualisation. The checks below verify schema, uniqueness, and submission alignment before any transformation is applied.

In [6]:
train_df = pd.read_csv(CFG.DATA_DIR / 'train.csv')
test_df = pd.read_csv(CFG.DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(CFG.DATA_DIR / 'sample_submission.csv')

required_train = {'id', 'text', 'listPrice'}
required_test = {'id', 'text'}
assert required_train.issubset(train_df.columns), train_df.columns.tolist()
assert required_test.issubset(test_df.columns), test_df.columns.tolist()
assert train_df['id'].is_unique and test_df['id'].is_unique
assert len(test_df) == len(sample_submission)
assert set(test_df['id']) == set(sample_submission['id'])

train_df['text'] = train_df['text'].fillna('').astype(str)
test_df['text'] = test_df['text'].fillna('').astype(str)
train_df['listPrice'] = pd.to_numeric(train_df['listPrice'], errors='raise').astype(float)

print('Train:', train_df.shape, '| Test:', test_df.shape)
print('Target range:', float(train_df['listPrice'].min()), 'to', float(train_df['listPrice'].max()))

Train: (14640, 3) | Test: (3659, 2)
Target range: 1.0 to 80000000.0


---

# Data Preparation and Feature Extraction <a name="3"></a>

---

The source text is kept semantically intact. Normalisation only repairs HTML/whitespace and removes commas inside numbers; punctuation, currency symbols, casing, and units are preserved. The `<NUM>` marker gives the Transformer an explicit local cue before each numeric expression without replacing the value itself.

## Text Normalisation

Numeric-aware text is generated once and shared by DeBERTa and TF-IDF. The original text remains available for regex feature extraction and failure analysis.

In [7]:
COMMA_INSIDE_NUMBER_RE = re.compile(r'(?<=\d),(?=\d)')
NUMERIC_TOKEN_RE = re.compile(r"(?<![\w>])([$£€]?\s*\d+(?:\.\d+)?(?:\s*[kKmM])?)(?!\w)")

def normalise_listing_text(text: str) -> str:
    value = html.unescape(str(text)).replace('\u00a0', ' ')
    value = COMMA_INSIDE_NUMBER_RE.sub('', value)
    value = re.sub(r'\s+', ' ', value).strip()
    if CFG.ADD_NUM_TOKEN:
        value = NUMERIC_TOKEN_RE.sub(lambda match: f"{CFG.NUM_TOKEN} {match.group(1).strip()}", value)
        value = re.sub(r'\s+', ' ', value).strip()
    return value

train_df['text_model'] = train_df['text'].map(normalise_listing_text)
test_df['text_model'] = test_df['text'].map(normalise_listing_text)

display(train_df[['id', 'text_model']].head(3))

,id,text_model
0,RE_009688,"Nestled in the heart of Randolph, [Redacted En..."
1,RE_015920,Beautiful end-unit townhouse in desirable Quin...
2,RE_014937,Lovely <NUM> 1925 bungalow! Forced air. Unfini...


#### Insights

> Numeric values are retained rather than collapsed to a generic placeholder. `<NUM>` therefore acts as a type cue while DeBERTa can still observe the original magnitude and nearby units.

## Structured Feature Extraction

The extractor targets quantities and cues that occur naturally in real-estate prose. Monetary values are deliberately represented as count/min/median/max features rather than assumed to be asking price, because descriptions also contain HOA dues, taxes, rent, credits, and renovation costs.

In [8]:
NUMBER_WORDS = {
    'zero': 0.0, 'one': 1.0, 'two': 2.0, 'three': 3.0, 'four': 4.0,
    'five': 5.0, 'six': 6.0, 'seven': 7.0, 'eight': 8.0, 'nine': 9.0,
    'ten': 10.0, 'eleven': 11.0, 'twelve': 12.0,
}
WORD_OR_NUMBER = r'(?:\d+(?:\.\d+)?|zero|one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve)'

BED_RE = re.compile(rf'\b(?P<n>{WORD_OR_NUMBER})\s*[- ]?(?:bed(?:room)?s?|br)\b', re.I)
BATH_RE = re.compile(rf'\b(?P<n>{WORD_OR_NUMBER})\s*[- ]?(?:full\s+|half\s+)?(?:bath(?:room)?s?|ba)\b', re.I)
SQFT_RE = re.compile(r'\b(?P<n>[\d,]+(?:\.\d+)?)\s*(?:sq\.?\s*ft\.?|sqft|square\s+feet|square\s+foot)\b', re.I)
ACRE_RE = re.compile(r'\b(?P<n>[\d,]+(?:\.\d+)?)\s*(?:acre|acres)\b', re.I)
GARAGE_RE = re.compile(rf'\b(?P<n>{WORD_OR_NUMBER})\s*[- ]?(?:car|stall)\s+garage\b', re.I)
CURRENCY_RE = re.compile(r'[$£€]\s*(?P<n>[\d,]+(?:\.\d+)?)(?P<suffix>[kKmM]?)')
HOA_RE = re.compile(r'(?:\bhoa\b|homeowners? association)(?:\s+(?:fee|fees|dues?))?[^$0-9]{0,24}[$]?\s*(?P<n>[\d,]+(?:\.\d+)?)', re.I)
YEAR_RE = re.compile(r'\b(?:18|19|20)\d{2}\b')
GENERIC_NUMBER_RE = re.compile(r'(?<!\w)[-+]?\d[\d,]*(?:\.\d+)?(?!\w)')

PROPERTY_TYPE_PATTERNS = [
    ('co_op', re.compile(r'\b(?:co-op|coop|cooperative)\b', re.I)),
    ('condo', re.compile(r'\b(?:condo|condominium)\b', re.I)),
    ('townhouse', re.compile(r'\b(?:townhouse|townhome)\b', re.I)),
    ('multi_family', re.compile(r'\b(?:multi[- ]family|two[- ]family|three[- ]family|duplex|triplex|fourplex)\b', re.I)),
    ('single_family', re.compile(r'\b(?:single[- ]family|detached home)\b', re.I)),
    ('manufactured', re.compile(r'\b(?:manufactured home|mobile home)\b', re.I)),
    ('land', re.compile(r'\b(?:vacant land|buildable lot|land parcel|acreage)\b', re.I)),
    ('apartment', re.compile(r'\bapartment\b', re.I)),
    ('estate', re.compile(r'\bestate\b', re.I)),
    ('ranch', re.compile(r'\branch\b', re.I)),
    ('bungalow', re.compile(r'\bbungalow\b', re.I)),
    ('victorian', re.compile(r'\bvictorian\b', re.I)),
]

KEYWORD_FLAGS = {
    'has_pool': r'\bpool\b',
    'has_waterfront': r'\b(?:waterfront|oceanfront|lakefront|riverfront)\b',
    'has_view': r'\b(?:views?|panoramic|cityscape|mountain view|ocean view)\b',
    'has_basement': r'\bbasement\b',
    'has_fireplace': r'\bfireplace\b',
    'has_new_construction': r'\b(?:new construction|newly built|brand[- ]new)\b',
    'has_renovated': r'\b(?:renovated|remodeled|updated|rehabbed)\b',
    'has_fixer': r'\b(?:fixer|fixer-upper|needs work|as-is|handyman)\b',
    'has_investment': r'\b(?:investment|investor|rental income|income producing)\b',
    'has_solar': r'\bsolar\b',
    'has_golf': r'\bgolf\b',
    'has_elevator': r'\belevator\b',
    'has_doorman': r'\b(?:doorman|concierge)\b',
    'has_hoa': r'\bhoa\b|homeowners? association',
    'has_school': r'\b(?:school|schools|school district)\b',
    'has_luxury': r'\b(?:luxury|luxurious|premier|prestigious|exclusive|penthouse)\b',
}
KEYWORD_FLAGS = {name: re.compile(pattern, re.I) for name, pattern in KEYWORD_FLAGS.items()}

In [9]:
def parse_number_token(token: str) -> float:
    value = str(token).strip().lower().replace(',', '')
    if value in NUMBER_WORDS:
        return NUMBER_WORDS[value]
    multiplier = 1.0
    if value.endswith('k'):
        multiplier = 1_000.0
        value = value[:-1]
    elif value.endswith('m'):
        multiplier = 1_000_000.0
        value = value[:-1]
    try:
        return float(value) * multiplier
    except ValueError:
        return np.nan

def plausible_max(values, low=0.0, high=np.inf):
    cleaned = [float(v) for v in values if pd.notna(v) and low <= float(v) <= high]
    return max(cleaned) if cleaned else np.nan

def plausible_min(values, low=0.0, high=np.inf):
    cleaned = [float(v) for v in values if pd.notna(v) and low <= float(v) <= high]
    return min(cleaned) if cleaned else np.nan

def extract_property_type(text: str) -> str:
    for label, pattern in PROPERTY_TYPE_PATTERNS:
        if pattern.search(text):
            return label
    return 'other'

def extract_structured_features(text: str) -> dict:
    raw = str(text)
    lowered = raw.lower()

    bed_values = [parse_number_token(match.group('n')) for match in BED_RE.finditer(raw)]
    bath_values = [parse_number_token(match.group('n')) for match in BATH_RE.finditer(raw)]
    sqft_values = [parse_number_token(match.group('n')) for match in SQFT_RE.finditer(raw)]
    acre_values = [parse_number_token(match.group('n')) for match in ACRE_RE.finditer(raw)]
    garage_values = [parse_number_token(match.group('n')) for match in GARAGE_RE.finditer(raw)]

    currency_values = []
    for match in CURRENCY_RE.finditer(raw):
        amount = parse_number_token(match.group('n'))
        suffix = match.group('suffix').lower()
        if suffix == 'k':
            amount *= 1_000.0
        elif suffix == 'm':
            amount *= 1_000_000.0
        if 0 <= amount <= 1_000_000_000:
            currency_values.append(amount)

    hoa_values = [parse_number_token(match.group('n')) for match in HOA_RE.finditer(raw)]
    hoa_values = [value for value in hoa_values if pd.notna(value) and 0 <= value <= 1_000_000]

    years = [int(value) for value in YEAR_RE.findall(raw)]
    years = [value for value in years if 1800 <= value <= 2035]

    generic_numbers = []
    for match in GENERIC_NUMBER_RE.finditer(COMMA_INSIDE_NUMBER_RE.sub('', raw)):
        value = parse_number_token(match.group(0))
        if pd.notna(value) and abs(value) <= 1_000_000_000:
            generic_numbers.append(abs(value))

    sentence_count = max(1, len(re.findall(r'[.!?]+', raw)))
    words = re.findall(r'\b\w+\b', raw)

    features = {
        'char_count': float(len(raw)),
        'word_count': float(len(words)),
        'sentence_count': float(sentence_count),
        'avg_word_length': float(np.mean([len(word) for word in words])) if words else 0.0,
        'digit_count': float(sum(character.isdigit() for character in raw)),
        'generic_number_count': float(len(generic_numbers)),
        'generic_number_max': max(generic_numbers) if generic_numbers else np.nan,
        'bedrooms': plausible_max(bed_values, 0, 30),
        'bathrooms': plausible_max(bath_values, 0, 30),
        'sqft': plausible_max(sqft_values, 50, 200_000),
        'lot_acres': plausible_max(acre_values, 0, 100_000),
        'garage_count': plausible_max(garage_values, 0, 30),
        'currency_count': float(len(currency_values)),
        'currency_min': min(currency_values) if currency_values else np.nan,
        'currency_median': float(np.median(currency_values)) if currency_values else np.nan,
        'currency_max': max(currency_values) if currency_values else np.nan,
        'hoa_amount': plausible_max(hoa_values, 0, 1_000_000),
        'year_count': float(len(years)),
        'year_min': min(years) if years else np.nan,
        'year_max': max(years) if years else np.nan,
        'property_type': extract_property_type(raw),
    }

    for name, pattern in KEYWORD_FLAGS.items():
        features[name] = int(bool(pattern.search(lowered)))

    return features

In [10]:
structured_train = train_df['text'].map(extract_structured_features).apply(pd.Series)
structured_test = test_df['text'].map(extract_structured_features).apply(pd.Series)

for frame in [structured_train, structured_test]:
    for source_col in ['sqft', 'lot_acres', 'currency_max', 'currency_median', 'hoa_amount', 'generic_number_max']:
        frame[f'log1p_{source_col}'] = np.log1p(frame[source_col].clip(lower=0))

structured_train.insert(0, 'id', train_df['id'].values)
structured_test.insert(0, 'id', test_df['id'].values)

structured_train.to_csv(CFG.ARTIFACT_DIR / 'structured_train.csv', index=False)
structured_test.to_csv(CFG.ARTIFACT_DIR / 'structured_test.csv', index=False)

coverage_cols = ['bedrooms', 'bathrooms', 'sqft', 'lot_acres', 'garage_count', 'currency_max', 'hoa_amount']
coverage = pd.DataFrame({
    'train_non_null_pct': structured_train[coverage_cols].notna().mean().mul(100),
    'test_non_null_pct': structured_test[coverage_cols].notna().mean().mul(100),
}).round(2)
display(coverage)
display(structured_train.head(3))

,train_non_null_pct,test_non_null_pct
bedrooms,51.15,56.85
bathrooms,40.63,44.17
sqft,16.89,16.18
lot_acres,13.91,12.30
garage_count,15.27,15.06
currency_max,7.23,8.09
hoa_amount,0.79,0.66


,id,char_count,word_count,sentence_count,avg_word_length,digit_count,generic_number_count,generic_number_max,bedrooms,bathrooms,...,has_doorman,has_hoa,has_school,has_luxury,log1p_sqft,log1p_lot_acres,log1p_currency_max,log1p_currency_median,log1p_hoa_amount,log1p_generic_number_max
0,RE_009688,1511.0,240.0,8.0,5.125000,0.0,0.0,NaN,NaN,NaN,...,0,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN
1,RE_015920,936.0,146.0,8.0,5.287671,1.0,1.0,2.0,2.0,NaN,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,1.098612
2,RE_014937,149.0,25.0,5.0,4.840000,5.0,1.0,1925.0,NaN,NaN,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,7.563201


#### Insights

> Coverage is expected to be partial: many listings omit explicit beds, baths, or square footage. CatBoost can retain missingness as signal, while the DeBERTa and TF-IDF branches still see the complete text.

> Currency features are intentionally agnostic about semantic role. The stacker can learn whether a monetary cue helps once it is combined with the semantic branches.

## Stratified Cross-Validation

The target is strongly right-skewed, so folds are stratified by quantiles of ranked `log1p(listPrice)`. Ranking avoids `qcut` failures when many listings share the same price. The same fold IDs are reused by every branch and by the second-level validation.

In [11]:
train_df['target_log'] = np.log1p(train_df['listPrice'].values)
ranked_target = pd.Series(train_df['target_log']).rank(method='first')
strat_labels = pd.qcut(ranked_target, q=CFG.STRAT_BINS, labels=False)

train_df['fold'] = -1
splitter = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
for fold, (_, valid_index) in enumerate(splitter.split(train_df, strat_labels)):
    train_df.loc[valid_index, 'fold'] = fold

assert (train_df['fold'] >= 0).all()
train_df[['id', 'listPrice', 'target_log', 'fold']].to_csv(CFG.ARTIFACT_DIR / 'folds.csv', index=False)

fold_summary = train_df.groupby('fold')['listPrice'].agg(['count', 'median', 'mean', 'max'])
display(fold_summary)
display(train_df['listPrice'].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_frame())

,count,median,mean,max
fold,,,,
0,2928,499900.0,817178.409836,54580000.0
1,2928,499900.0,889868.317281,80000000.0
2,2928,499900.0,856303.665984,37500000.0
3,2928,499900.0,831638.469945,38500000.0
4,2928,499900.0,800375.730533,32000000.0


,listPrice
count,1.464000e+04
mean,8.390729e+05
std,1.912897e+06
min,1.000000e+00
50%,4.999000e+05
90%,1.449000e+06
95%,2.300000e+06
99%,6.561000e+06
max,8.000000e+07


#### Insights

> Quantile stratification keeps ordinary and luxury regimes represented in every validation fold. This is important because MAE in original dollars is dominated by a relatively small number of expensive listings.

## Shared Evaluation Helpers

All branches predict log-price internally. The helper below converts predictions safely back to the competition scale and reports both overall and luxury-tail error.

In [12]:
TAIL_THRESHOLD = float(train_df['listPrice'].quantile(0.90))

def log_to_price(pred_log):
    clipped = np.clip(np.asarray(pred_log, dtype=float), 0.0, np.log1p(CFG.MAX_PRICE_PRED))
    return np.expm1(clipped)

def regression_metrics(y_true_price, pred_log, tail_threshold=TAIL_THRESHOLD):
    y_true_price = np.asarray(y_true_price, dtype=float)
    pred_log = np.asarray(pred_log, dtype=float).reshape(-1)
    pred_price = log_to_price(pred_log)
    y_true_log = np.log1p(y_true_price)
    tail_mask = y_true_price >= tail_threshold
    return {
        'mae_original': mean_absolute_error(y_true_price, pred_price),
        'mae_log': mean_absolute_error(y_true_log, pred_log),
        'tail_mae': mean_absolute_error(y_true_price[tail_mask], pred_price[tail_mask]),
    }

print('Top-decile threshold:', f'{TAIL_THRESHOLD:,.0f}')

Top-decile threshold: 1,449,000


---

# Domain-Adaptive Pretraining <a name="4"></a>

---

The DeBERTa encoder is adapted to real-estate language before supervised regression. The default follows the NB-1 plan and uses both train and test descriptions without labels for masked-language modelling. If competition policy forbids transductive unsupervised adaptation, set `DAPT_USE_TEST_TEXT=False` before running this section.

## Build the Numeric-Aware Tokenizer

A single additional special token marks numeric expressions. The token embedding is learned during continued pretraining, while the original number tokens remain in the sequence.

In [13]:
tokenizer = AutoTokenizer.from_pretrained(CFG.BASE_MODEL, use_fast=True)
if CFG.ADD_NUM_TOKEN and CFG.NUM_TOKEN not in tokenizer.get_vocab():
    tokenizer.add_special_tokens({'additional_special_tokens': [CFG.NUM_TOKEN]})

print('Tokenizer size:', len(tokenizer), '| NUM token id:', tokenizer.convert_tokens_to_ids(CFG.NUM_TOKEN))

Tokenizer size: 128002 | NUM token id: 128001


/venv/main/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


## Continue Pretraining with MLM

The corpus is tokenised to 512 tokens and trained with 15% dynamic masking. On a modern RTX 5090, BF16 is preferred when available; gradient accumulation yields an effective batch size of 64 with the default settings. Existing DAPT artifacts are reused unless `FORCE_DAPT=True`.

In [14]:
dapt_model = AutoModelForMaskedLM.from_pretrained(CFG.BASE_MODEL)
dapt_model.resize_token_embeddings(len(tokenizer))

mlm_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=CFG.DAPT_MLM_PROBABILITY,
)

dapt_args = TrainingArguments(
    output_dir=str(CFG.DAPT_DIR),
    overwrite_output_dir=True,
    num_train_epochs=CFG.DAPT_EPOCHS,
    per_device_train_batch_size=CFG.DAPT_BATCH_SIZE,
    gradient_accumulation_steps=CFG.DAPT_GRAD_ACCUM,
    learning_rate=CFG.DAPT_LR,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type='linear',
    bf16=USE_BF16,
    fp16=USE_FP16,
    gradient_checkpointing=CFG.GRADIENT_CHECKPOINTING,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=50,
    save_strategy='epoch',
    save_total_limit=2,
    dataloader_num_workers=CFG.NUM_WORKERS,
    report_to='none',
    seed=CFG.SEED,
    data_seed=CFG.SEED,
)

Some weights of DebertaV2ForMaskedLM were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


#### Insights

> DAPT is separated from supervised folds because it never sees `listPrice`. The adapted checkpoint is therefore shared across all folds, avoiding five redundant MLM runs.

---

# DeBERTa Regression Fine-Tuning <a name="5"></a>

---

Each supervised fold starts from the same domain-adapted encoder and predicts `log1p(listPrice)`. Validation predictions are written into a single OOF vector, while every fold also predicts the full test set; averaging those test predictions provides the branch-level test estimate.

## Regression Dataset and Metrics

The Hugging Face dataset stores float labels under the `labels` field. Competition-scale MAE is computed by exponentiating the model output inside `compute_metrics`, so early stopping tracks the metric that matters.

In [15]:
dapt_ready = (CFG.DAPT_DIR / 'config.json').exists() and any(CFG.DAPT_DIR.glob('model*.safetensors'))

MODEL_SOURCE = str(CFG.DAPT_DIR) if dapt_ready else CFG.BASE_MODEL
reg_tokenizer = AutoTokenizer.from_pretrained(MODEL_SOURCE, use_fast=True)

if CFG.ADD_NUM_TOKEN and CFG.NUM_TOKEN not in reg_tokenizer.get_vocab():
    reg_tokenizer.add_special_tokens({'additional_special_tokens': [CFG.NUM_TOKEN]})

def make_regression_dataset(frame: pd.DataFrame, include_labels: bool = True):
    payload = {'text': frame['text_model'].tolist()}
    if include_labels:
        payload['labels'] = frame['target_log'].astype(np.float32).tolist()

    dataset = Dataset.from_dict(payload)
    remove_columns = ['text']

    tokenized = dataset.map(
        lambda batch: reg_tokenizer(
            batch['text'],
            truncation=True,
            max_length=CFG.FT_MAX_LENGTH,
        ),
        batched=True,
        remove_columns=remove_columns,
        desc='Tokenising regression data',
    )
    return tokenized

def hf_regression_metrics(eval_prediction):
    pred_log, true_log = eval_prediction
    pred_log = np.asarray(pred_log).reshape(-1)
    true_log = np.asarray(true_log).reshape(-1)
    true_price = np.expm1(true_log)
    pred_price = log_to_price(pred_log)
    return {
        'mae_original': mean_absolute_error(true_price, pred_price),
        'mae_log': mean_absolute_error(true_log, pred_log),
    }

test_hf = make_regression_dataset(test_df, include_labels=False)

print('DAPT ready:', dapt_ready)
print('Model source:', MODEL_SOURCE, '| Test tokenised:', len(test_hf))

/venv/main/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Tokenising regression data:   0%|          | 0/3659 [00:00<?, ? examples/s]

DAPT ready: False
Model source: microsoft/deberta-v3-base | Test tokenised: 3659


## Five-Fold Fine-Tuning

The default per-device batch size is 16 with four gradient-accumulation steps, producing an effective batch of 64. If memory is insufficient, reduce `FT_BATCH_SIZE` while increasing `FT_GRAD_ACCUM`; no other cell needs to change.

In [16]:
deberta_oof_path = CFG.ARTIFACT_DIR / 'deberta_oof.csv'
deberta_test_path = CFG.ARTIFACT_DIR / 'deberta_test.csv'

if CFG.RUN_DEBERTA:
    deberta_oof_log = np.full(len(train_df), np.nan, dtype=np.float64)
    deberta_test_log_folds = []
    deberta_fold_metrics = []

    for fold in CFG.FOLDS_TO_RUN:
        print(f'\n===== DeBERTa fold {fold}/{CFG.N_FOLDS - 1} =====')

        train_mask = train_df['fold'].values != fold
        valid_mask = train_df['fold'].values == fold

        fold_train = train_df.loc[train_mask].reset_index(drop=True)
        fold_valid = train_df.loc[valid_mask].reset_index(drop=True)

        train_hf = make_regression_dataset(fold_train, include_labels=True)
        valid_hf = make_regression_dataset(fold_valid, include_labels=True)

        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_SOURCE,
            num_labels=1,
            ignore_mismatched_sizes=True,
        )

        model.config.problem_type = 'regression'
        model.resize_token_embeddings(len(reg_tokenizer))

        fold_dir = CFG.FOLD_MODEL_DIR / f'fold_{fold}'
        fold_dir.mkdir(parents=True, exist_ok=True)

        args = TrainingArguments(
            output_dir=str(fold_dir / 'checkpoints'),

            num_train_epochs=CFG.FT_EPOCHS,

            per_device_train_batch_size=CFG.FT_BATCH_SIZE,
            per_device_eval_batch_size=CFG.FT_EVAL_BATCH_SIZE,
            gradient_accumulation_steps=CFG.FT_GRAD_ACCUM,

            learning_rate=CFG.FT_LR,
            weight_decay=CFG.FT_WEIGHT_DECAY,
            warmup_ratio=CFG.FT_WARMUP_RATIO,
            lr_scheduler_type='linear',

            eval_strategy='epoch',
            save_strategy='epoch',

            logging_strategy='steps',
            logging_steps=50,

            load_best_model_at_end=True,
            metric_for_best_model='mae_original',
            greater_is_better=False,
            save_total_limit=1,

            bf16=USE_BF16,
            fp16=USE_FP16,

            gradient_checkpointing=CFG.GRADIENT_CHECKPOINTING,
            gradient_checkpointing_kwargs=(
                {"use_reentrant": False}
                if CFG.GRADIENT_CHECKPOINTING
                else None
            ),

            dataloader_num_workers=CFG.NUM_WORKERS,
            report_to='none',

            seed=CFG.SEED + fold,
            data_seed=CFG.SEED + fold,
        )

        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=train_hf,
            eval_dataset=valid_hf,
            compute_metrics=hf_regression_metrics,
            callbacks=[
                EarlyStoppingCallback(
                    early_stopping_patience=CFG.FT_EARLY_STOPPING_PATIENCE
                )
            ],
            processing_class=reg_tokenizer,
        )

        trainer.train()

        valid_pred_log = np.asarray(
            trainer.predict(valid_hf).predictions
        ).reshape(-1)

        test_pred_log = np.asarray(
            trainer.predict(test_hf).predictions
        ).reshape(-1)

        valid_indices = train_df.index[valid_mask]
        deberta_oof_log[valid_indices] = valid_pred_log
        deberta_test_log_folds.append(test_pred_log)

        metrics = regression_metrics(
            fold_valid['listPrice'].values,
            valid_pred_log,
        )
        metrics['fold'] = int(fold)
        deberta_fold_metrics.append(metrics)

        print(metrics)

        best_model_dir = fold_dir / 'best_model'

        trainer.save_model(str(best_model_dir))
        reg_tokenizer.save_pretrained(str(best_model_dir))

        pd.DataFrame({
            'id': fold_valid['id'].values,
            'fold': fold,
            'pred_log': valid_pred_log,
            'pred_price': log_to_price(valid_pred_log),
        }).to_csv(
            fold_dir / 'valid_predictions.csv',
            index=False,
        )

        pd.DataFrame({
            'id': test_df['id'].values,
            'fold': fold,
            'pred_log': test_pred_log,
            'pred_price': log_to_price(test_pred_log),
        }).to_csv(
            fold_dir / 'test_predictions.csv',
            index=False,
        )

        del trainer, model, train_hf, valid_hf

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if np.isnan(deberta_oof_log).any():
        missing_folds = sorted(
            train_df.loc[
                np.isnan(deberta_oof_log),
                'fold',
            ].unique().tolist()
        )
        raise RuntimeError(
            f'DeBERTa OOF incomplete. Missing folds: {missing_folds}'
        )

    deberta_test_log = np.mean(
        np.vstack(deberta_test_log_folds),
        axis=0,
    )

    pd.DataFrame({
        'id': train_df['id'],
        'fold': train_df['fold'],
        'pred_log': deberta_oof_log,
        'pred_price': log_to_price(deberta_oof_log),
    }).to_csv(
        deberta_oof_path,
        index=False,
    )

    pd.DataFrame({
        'id': test_df['id'],
        'pred_log': deberta_test_log,
        'pred_price': log_to_price(deberta_test_log),
    }).to_csv(
        deberta_test_path,
        index=False,
    )

    pd.DataFrame(
        deberta_fold_metrics
    ).to_csv(
        CFG.ARTIFACT_DIR / 'deberta_fold_metrics.csv',
        index=False,
    )

else:
    if not deberta_oof_path.exists() or not deberta_test_path.exists():
        raise FileNotFoundError(
            'RUN_DEBERTA=False but saved DeBERTa predictions were not found.'
        )

    deberta_oof_log = (
        pd.read_csv(deberta_oof_path)
        .set_index('id')
        .loc[train_df['id'], 'pred_log']
        .values
    )

    deberta_test_log = (
        pd.read_csv(deberta_test_path)
        .set_index('id')
        .loc[test_df['id'], 'pred_log']
        .values
    )

deberta_metrics = regression_metrics(
    train_df['listPrice'].values,
    deberta_oof_log,
)

print('DeBERTa OOF:', deberta_metrics)


===== DeBERTa fold 0/4 =====


Tokenising regression data:   0%|          | 0/11712 [00:00<?, ? examples/s]

Tokenising regression data:   0%|          | 0/2928 [00:00<?, ? examples/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Mae Original,Mae Log
1,4.508100,1.115023,672148.952929,0.822421
2,0.744900,1.028583,861484.396801,0.809754


...

Tokenising regression data:   0%|          | 0/11712 [00:00<?, ? examples/s]

Tokenising regression data:   0%|          | 0/2928 [00:00<?, ? examples/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Mae Original,Mae Log
1,2.960900,1.107265,644123.041844,0.841686
2,0.677600,0.971460,770132.026799,0.826668


{'mae_original': 644123.0307534422, 'mae_log': 0.8416858922840926, 'tail_mae': 2071736.2843617864, 'fold': 4}
DeBERTa OOF: {'mae_original': 594574.7196823688, 'mae_log': 0.7169085008352991, 'tail_mae': 2057785.5260681277}


#### Insights

> The OOF vector is the only DeBERTa signal passed to the stacker. Test predictions are the arithmetic mean of fold-level log predictions, keeping the train/test construction symmetric.

---

# TF-IDF + Ridge Branch <a name="6"></a>

---

Character n-grams retain spelling variants, abbreviations, numbers, and partial location/entity tokens, while word uni/bi-grams preserve interpretable phrases. Ridge regression on log-price provides a strong low-variance complement to the Transformer.

## Cross-Validated Word + Character TF-IDF

Each fold fits its own vectorizers using only that fold's training rows. This is more expensive than fitting TF-IDF once globally, but keeps OOF predictions strictly out-of-sample. Feature caps keep memory use practical on a typical Vast.ai instance.

In [17]:
tfidf_oof_path = CFG.ARTIFACT_DIR / 'tfidf_oof.csv'
tfidf_test_path = CFG.ARTIFACT_DIR / 'tfidf_test.csv'

if CFG.RUN_TFIDF:
    tfidf_oof_log = np.zeros(len(train_df), dtype=np.float64)
    tfidf_test_log_folds = []
    tfidf_fold_metrics = []

    for fold in range(CFG.N_FOLDS):
        print(f'\n===== TF-IDF fold {fold}/{CFG.N_FOLDS - 1} =====')
        train_mask = train_df['fold'].values != fold
        valid_mask = train_df['fold'].values == fold

        word_vectorizer = TfidfVectorizer(
            lowercase=True,
            strip_accents='unicode',
            ngram_range=(1, 2),
            min_df=CFG.TFIDF_WORD_MIN_DF,
            max_df=0.995,
            max_features=CFG.TFIDF_WORD_MAX_FEATURES,
            sublinear_tf=True,
            dtype=np.float32,
        )
        char_vectorizer = TfidfVectorizer(
            analyzer='char_wb',
            lowercase=True,
            ngram_range=(3, 5),
            min_df=CFG.TFIDF_CHAR_MIN_DF,
            max_features=CFG.TFIDF_CHAR_MAX_FEATURES,
            sublinear_tf=True,
            dtype=np.float32,
        )

        text_train = train_df.loc[train_mask, 'text_model'].values
        text_valid = train_df.loc[valid_mask, 'text_model'].values
        y_train_log = train_df.loc[train_mask, 'target_log'].values

        xw_train = word_vectorizer.fit_transform(text_train)
        xc_train = char_vectorizer.fit_transform(text_train)
        x_train = hstack([xw_train, xc_train], format='csr')

        xw_valid = word_vectorizer.transform(text_valid)
        xc_valid = char_vectorizer.transform(text_valid)
        x_valid = hstack([xw_valid, xc_valid], format='csr')

        xw_test = word_vectorizer.transform(test_df['text_model'].values)
        xc_test = char_vectorizer.transform(test_df['text_model'].values)
        x_test = hstack([xw_test, xc_test], format='csr')

        ridge = Ridge(alpha=CFG.TFIDF_RIDGE_ALPHA, solver='lsqr')
        ridge.fit(x_train, y_train_log)
        valid_pred_log = ridge.predict(x_valid)
        test_pred_log = ridge.predict(x_test)

        tfidf_oof_log[valid_mask] = valid_pred_log
        tfidf_test_log_folds.append(test_pred_log)
        metrics = regression_metrics(train_df.loc[valid_mask, 'listPrice'].values, valid_pred_log)
        metrics['fold'] = fold
        tfidf_fold_metrics.append(metrics)
        print(metrics)

        if CFG.SAVE_CLASSICAL_FOLDS:
            joblib.dump(
                {'word_vectorizer': word_vectorizer, 'char_vectorizer': char_vectorizer, 'ridge': ridge},
                CFG.ARTIFACT_DIR / f'tfidf_ridge_fold_{fold}.joblib',
                compress=3,
            )

        del xw_train, xc_train, x_train, xw_valid, xc_valid, x_valid, xw_test, xc_test, x_test
        del word_vectorizer, char_vectorizer, ridge
        gc.collect()

    tfidf_test_log = np.mean(np.vstack(tfidf_test_log_folds), axis=0)
    pd.DataFrame({
        'id': train_df['id'],
        'fold': train_df['fold'],
        'pred_log': tfidf_oof_log,
        'pred_price': log_to_price(tfidf_oof_log),
    }).to_csv(tfidf_oof_path, index=False)
    pd.DataFrame({
        'id': test_df['id'],
        'pred_log': tfidf_test_log,
        'pred_price': log_to_price(tfidf_test_log),
    }).to_csv(tfidf_test_path, index=False)
    pd.DataFrame(tfidf_fold_metrics).to_csv(CFG.ARTIFACT_DIR / 'tfidf_fold_metrics.csv', index=False)
else:
    if not tfidf_oof_path.exists() or not tfidf_test_path.exists():
        raise FileNotFoundError('RUN_TFIDF=False but saved TF-IDF predictions were not found.')
    tfidf_oof_log = pd.read_csv(tfidf_oof_path).set_index('id').loc[train_df['id'], 'pred_log'].values
    tfidf_test_log = pd.read_csv(tfidf_test_path).set_index('id').loc[test_df['id'], 'pred_log'].values

tfidf_metrics = regression_metrics(train_df['listPrice'].values, tfidf_oof_log)
print('TF-IDF OOF:', tfidf_metrics)


===== TF-IDF fold 0/4 =====
{'mae_original': 370534.25399260386, 'mae_log': 0.4515581596445754, 'tail_mae': 2152647.6778806443, 'fold': 0}

===== TF-IDF fold 1/4 =====
{'mae_original': 437169.48677566723, 'mae_log': 0.4465697106874063, 'tail_mae': 2827130.624802048, 'fold': 1}

===== TF-IDF fold 2/4 =====
{'mae_original': 401457.5970041542, 'mae_log': 0.4506571095637316, 'tail_mae': 2470810.7018712247, 'fold': 2}

===== TF-IDF fold 3/4 =====
{'mae_original': 382898.2853019972, 'mae_log': 0.44910808258813134, 'tail_mae': 2296666.073506725, 'fold': 3}

===== TF-IDF fold 4/4 =====
{'mae_original': 357992.78274386196, 'mae_log': 0.45130056504035526, 'tail_mae': 2013246.6850309316, 'fold': 4}
TF-IDF OOF: {'mae_original': 390010.4811636569, 'mae_log': 0.44983872550484, 'tail_mae': 2352386.375790143}


#### Insights

> This branch is deliberately very different from DeBERTa: it does not need deep semantic composition, but it can memorise fine-grained lexical, abbreviation, entity, and numeric surface patterns that may be diluted in a Transformer representation.

## Optional Nearest-Neighbour Sanity Check

The NB-1 plan proposes a text-retrieval lower bound. It is disabled by default because it does not contribute to the final stack, but it can reveal near-duplicate or highly localised listing patterns.

In [18]:
if CFG.RUN_KNN_BASELINE:
    knn_vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=120_000,
        sublinear_tf=True,
        dtype=np.float32,
    )
    x_all = knn_vectorizer.fit_transform(train_df['text_model'])
    knn = NearestNeighbors(n_neighbors=CFG.KNN_K + 1, metric='cosine', algorithm='brute')
    knn.fit(x_all)
    distances, indices = knn.kneighbors(x_all)
    neighbour_indices = indices[:, 1:]
    neighbour_distances = distances[:, 1:]
    weights = 1.0 / np.clip(neighbour_distances, 1e-3, None)
    neighbour_prices = train_df['listPrice'].values[neighbour_indices]
    knn_pred_price = (weights * neighbour_prices).sum(axis=1) / weights.sum(axis=1)
    print('KNN self-excluded sanity MAE:', mean_absolute_error(train_df['listPrice'], knn_pred_price))
    del x_all, knn, knn_vectorizer
    gc.collect()
else:
    print('KNN baseline disabled.')

KNN baseline disabled.


---

# Structured CatBoost Branch <a name="7"></a>

---

The structured branch treats extracted quantities and explicit property cues as a compact tabular problem. It is intentionally trained independently from TF-IDF and DeBERTa so the final stack can exploit disagreement among the branches.

## Cross-Validated CatBoost

CatBoost handles missing numerical values and the derived `property_type` category directly. CPU mode is the default because the dataset is small and it leaves the GPU free for Transformer work; set `CATBOOST_TASK_TYPE='GPU'` if desired.

In [19]:
structured_oof_path = CFG.ARTIFACT_DIR / 'structured_oof.csv'
structured_test_path = CFG.ARTIFACT_DIR / 'structured_test_predictions.csv'
structured_feature_cols = [column for column in structured_train.columns if column != 'id']
structured_cat_cols = ['property_type']

if CFG.RUN_STRUCTURED:
    structured_oof_log = np.zeros(len(train_df), dtype=np.float64)
    structured_test_log_folds = []
    structured_fold_metrics = []

    x_struct_train = structured_train[structured_feature_cols].copy()
    x_struct_test = structured_test[structured_feature_cols].copy()
    x_struct_train['property_type'] = x_struct_train['property_type'].fillna('other').astype(str)
    x_struct_test['property_type'] = x_struct_test['property_type'].fillna('other').astype(str)

    for fold in range(CFG.N_FOLDS):
        print(f'\n===== Structured fold {fold}/{CFG.N_FOLDS - 1} =====')
        train_mask = train_df['fold'].values != fold
        valid_mask = train_df['fold'].values == fold

        model = CatBoostRegressor(
            iterations=CFG.CATBOOST_ITERATIONS,
            depth=CFG.CATBOOST_DEPTH,
            learning_rate=CFG.CATBOOST_LR,
            l2_leaf_reg=CFG.CATBOOST_L2,
            loss_function='RMSE',
            eval_metric='MAE',
            random_seed=CFG.SEED + fold,
            task_type=CFG.CATBOOST_TASK_TYPE,
            allow_writing_files=False,
            verbose=200,
        )
        model.fit(
            x_struct_train.loc[train_mask],
            train_df.loc[train_mask, 'target_log'],
            eval_set=(x_struct_train.loc[valid_mask], train_df.loc[valid_mask, 'target_log']),
            cat_features=structured_cat_cols,
            early_stopping_rounds=CFG.CATBOOST_EARLY_STOP,
            use_best_model=True,
        )

        valid_pred_log = model.predict(x_struct_train.loc[valid_mask])
        test_pred_log = model.predict(x_struct_test)
        structured_oof_log[valid_mask] = valid_pred_log
        structured_test_log_folds.append(test_pred_log)

        metrics = regression_metrics(train_df.loc[valid_mask, 'listPrice'].values, valid_pred_log)
        metrics['fold'] = fold
        metrics['best_iteration'] = int(model.get_best_iteration())
        structured_fold_metrics.append(metrics)
        print(metrics)
        model.save_model(str(CFG.ARTIFACT_DIR / f'structured_catboost_fold_{fold}.cbm'))

    structured_test_log = np.mean(np.vstack(structured_test_log_folds), axis=0)
    pd.DataFrame({
        'id': train_df['id'],
        'fold': train_df['fold'],
        'pred_log': structured_oof_log,
        'pred_price': log_to_price(structured_oof_log),
    }).to_csv(structured_oof_path, index=False)
    pd.DataFrame({
        'id': test_df['id'],
        'pred_log': structured_test_log,
        'pred_price': log_to_price(structured_test_log),
    }).to_csv(structured_test_path, index=False)
    pd.DataFrame(structured_fold_metrics).to_csv(CFG.ARTIFACT_DIR / 'structured_fold_metrics.csv', index=False)
else:
    if not structured_oof_path.exists() or not structured_test_path.exists():
        raise FileNotFoundError('RUN_STRUCTURED=False but saved structured predictions were not found.')
    structured_oof_log = pd.read_csv(structured_oof_path).set_index('id').loc[train_df['id'], 'pred_log'].values
    structured_test_log = pd.read_csv(structured_test_path).set_index('id').loc[test_df['id'], 'pred_log'].values

structured_metrics = regression_metrics(train_df['listPrice'].values, structured_oof_log)
print('Structured OOF:', structured_metrics)


===== Structured fold 0/4 =====
0:	learn: 0.7004138	test: 0.7030368	best: 0.7030368 (0)	total: 67.3ms	remaining: 3m 21s
200:	learn: 0.5574201	test: 0.5820402	best: 0.5820402 (200)	total: 1.98s	remaining: 27.5s
400:	learn: 0.5332553	test: 0.5698830	best: 0.5698830 (400)	total: 3.92s	remaining: 25.4s
600:	learn: 0.5184326	test: 0.5656406	best: 0.5656321 (597)	total: 6.14s	remaining: 24.5s
800:	learn: 0.5058197	test: 0.5630710	best: 0.5630544 (799)	total: 8.05s	remaining: 22.1s
1000:	learn: 0.4945373	test: 0.5610516	best: 0.5609799 (995)	total: 10.4s	remaining: 20.8s
1200:	learn: 0.4847124	test: 0.5603457	best: 0.5603414 (1198)	total: 12s	remaining: 18s
1400:	learn: 0.4751516	test: 0.5596012	best: 0.5595180 (1378)	total: 13.8s	remaining: 15.8s
1600:	learn: 0.4671511	test: 0.5595650	best: 0.5594296 (1576)	total: 15.4s	remaining: 13.5s
1800:	learn: 0.4595984	test: 0.5594149	best: 0.5593348 (1645)	total: 17.3s	remaining: 11.5s
Stopped by overfitting detector  (200 iterations wait)

bestTest

#### Insights

> The structured model is not expected to dominate when explicit quantities are absent. Its value is complementary: when a listing states `4 bedrooms`, `3.5 baths`, `2,500 sq ft`, acreage, HOA fees, or luxury cues, those values enter the ensemble without relying on the Transformer to perform implicit numeric parsing.

---

# Late Fusion / Stacking <a name="8"></a>

---

Stacking uses only OOF base predictions on training rows. Two meta-models are tested under the same folds: a regularised linear Ridge stack and a shallow CatBoost stack. The notebook selects the lower OOF MAE automatically, then refits that stacker on all OOF meta-data and applies it to averaged base test predictions.

## Assemble Meta-Features

The three branch predictions are supplemented with a small set of low-risk numeric cues. These extra columns let the stacker condition branch trust on property scale without duplicating the entire structured model.

In [20]:
meta_numeric_cols = [
    'bedrooms', 'bathrooms', 'garage_count', 'word_count',
    'log1p_sqft', 'log1p_lot_acres', 'log1p_currency_max',
    'log1p_hoa_amount', 'has_luxury', 'has_fixer', 'has_waterfront',
]

meta_train = pd.DataFrame({
    'deberta_log': deberta_oof_log,
    'tfidf_log': tfidf_oof_log,
    'structured_log': structured_oof_log,
})
meta_test = pd.DataFrame({
    'deberta_log': deberta_test_log,
    'tfidf_log': tfidf_test_log,
    'structured_log': structured_test_log,
})

for column in meta_numeric_cols:
    meta_train[column] = structured_train[column].values
    meta_test[column] = structured_test[column].values

display(meta_train.head())
display(meta_train[['deberta_log', 'tfidf_log', 'structured_log']].corr())

,deberta_log,tfidf_log,structured_log,bedrooms,bathrooms,garage_count,word_count,log1p_sqft,log1p_lot_acres,log1p_currency_max,log1p_hoa_amount,has_luxury,has_fixer,has_waterfront
0,13.8750,12.716695,13.254411,NaN,NaN,NaN,240.0,NaN,NaN,NaN,NaN,0,0,0
1,14.1250,12.885845,12.975086,2.0,NaN,NaN,146.0,NaN,NaN,NaN,NaN,0,0,0
2,13.1250,12.813144,13.004704,NaN,NaN,NaN,25.0,NaN,NaN,NaN,NaN,0,0,0
3,13.2500,12.728367,12.586655,NaN,NaN,NaN,48.0,NaN,NaN,NaN,NaN,0,0,0
4,13.3125,12.866248,13.003562,NaN,NaN,1.0,144.0,NaN,NaN,NaN,NaN,0,0,0


,deberta_log,tfidf_log,structured_log
deberta_log,1.000000,0.835386,0.703817
tfidf_log,0.835386,1.000000,0.773221
structured_log,0.703817,0.773221,1.000000


## Honest Second-Level Cross-Validation

Each validation row is scored by a meta-model that was fit only on OOF predictions from other rows. This produces a meaningful stacking estimate rather than evaluating a meta-model on the same OOF table used to fit it.

In [21]:
stack_oof_path = CFG.ARTIFACT_DIR / 'stack_oof.csv'
stack_test_path = CFG.ARTIFACT_DIR / 'stack_test.csv'
stack_selection_path = CFG.ARTIFACT_DIR / 'stack_selection.json'

if CFG.RUN_STACKING:
    ridge_stack_oof = np.zeros(len(train_df), dtype=np.float64)
    cat_stack_oof = np.zeros(len(train_df), dtype=np.float64)
    cat_best_iterations = []

    for fold in range(CFG.N_FOLDS):
        train_mask = train_df['fold'].values != fold
        valid_mask = train_df['fold'].values == fold

        ridge_stack = Pipeline([
            ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
            ('scaler', StandardScaler()),
            ('ridge', Ridge(alpha=CFG.META_RIDGE_ALPHA)),
        ])
        ridge_stack.fit(meta_train.loc[train_mask], train_df.loc[train_mask, 'target_log'])
        ridge_stack_oof[valid_mask] = ridge_stack.predict(meta_train.loc[valid_mask])

        cat_stack = CatBoostRegressor(
            iterations=CFG.META_CATBOOST_ITERATIONS,
            depth=CFG.META_CATBOOST_DEPTH,
            learning_rate=CFG.META_CATBOOST_LR,
            l2_leaf_reg=8.0,
            loss_function='RMSE',
            eval_metric='MAE',
            random_seed=CFG.SEED + 100 + fold,
            task_type='CPU',
            allow_writing_files=False,
            verbose=False,
        )
        cat_stack.fit(
            meta_train.loc[train_mask],
            train_df.loc[train_mask, 'target_log'],
            eval_set=(meta_train.loc[valid_mask], train_df.loc[valid_mask, 'target_log']),
            early_stopping_rounds=CFG.META_EARLY_STOP,
            use_best_model=True,
        )
        cat_stack_oof[valid_mask] = cat_stack.predict(meta_train.loc[valid_mask])
        cat_best_iterations.append(max(50, int(cat_stack.get_best_iteration()) + 1))

    candidate_metrics = {
        'ridge_stack': regression_metrics(train_df['listPrice'].values, ridge_stack_oof),
        'catboost_stack': regression_metrics(train_df['listPrice'].values, cat_stack_oof),
        'mean_log_blend': regression_metrics(
            train_df['listPrice'].values,
            meta_train[['deberta_log', 'tfidf_log', 'structured_log']].mean(axis=1).values,
        ),
    }
    candidate_table = pd.DataFrame(candidate_metrics).T.sort_values('mae_original')
    display(candidate_table)

    best_stacker = candidate_table.index[0]
    if best_stacker == 'mean_log_blend':
        stack_oof_log = meta_train[['deberta_log', 'tfidf_log', 'structured_log']].mean(axis=1).values
        stack_test_log = meta_test[['deberta_log', 'tfidf_log', 'structured_log']].mean(axis=1).values
    elif best_stacker == 'ridge_stack':
        stack_oof_log = ridge_stack_oof
        final_meta_model = Pipeline([
            ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
            ('scaler', StandardScaler()),
            ('ridge', Ridge(alpha=CFG.META_RIDGE_ALPHA)),
        ])
        final_meta_model.fit(meta_train, train_df['target_log'])
        stack_test_log = final_meta_model.predict(meta_test)
        joblib.dump(final_meta_model, CFG.ARTIFACT_DIR / 'meta_ridge.joblib')
    else:
        stack_oof_log = cat_stack_oof
        final_iterations = int(np.median(cat_best_iterations))
        final_meta_model = CatBoostRegressor(
            iterations=final_iterations,
            depth=CFG.META_CATBOOST_DEPTH,
            learning_rate=CFG.META_CATBOOST_LR,
            l2_leaf_reg=8.0,
            loss_function='RMSE',
            random_seed=CFG.SEED + 999,
            task_type='CPU',
            allow_writing_files=False,
            verbose=False,
        )
        final_meta_model.fit(meta_train, train_df['target_log'])
        stack_test_log = final_meta_model.predict(meta_test)
        final_meta_model.save_model(str(CFG.ARTIFACT_DIR / 'meta_catboost.cbm'))

    pd.DataFrame({
        'id': train_df['id'],
        'fold': train_df['fold'],
        'pred_log': stack_oof_log,
        'pred_price': log_to_price(stack_oof_log),
    }).to_csv(stack_oof_path, index=False)
    pd.DataFrame({
        'id': test_df['id'],
        'pred_log': stack_test_log,
        'pred_price': log_to_price(stack_test_log),
    }).to_csv(stack_test_path, index=False)

    selection = {
        'best_stacker': best_stacker,
        'candidate_metrics': candidate_table.to_dict(orient='index'),
    }
    stack_selection_path.write_text(json.dumps(selection, indent=2), encoding='utf-8')
else:
    if not stack_oof_path.exists() or not stack_test_path.exists():
        raise FileNotFoundError('RUN_STACKING=False but saved stacking predictions were not found.')
    stack_oof_log = pd.read_csv(stack_oof_path).set_index('id').loc[train_df['id'], 'pred_log'].values
    stack_test_log = pd.read_csv(stack_test_path).set_index('id').loc[test_df['id'], 'pred_log'].values
    best_stacker = json.loads(stack_selection_path.read_text(encoding='utf-8'))['best_stacker'] if stack_selection_path.exists() else 'saved'

stack_metrics = regression_metrics(train_df['listPrice'].values, stack_oof_log)
print('Selected stacker:', best_stacker)
print('Stack OOF:', stack_metrics)

,mae_original,mae_log,tail_mae
catboost_stack,359771.821807,0.429977,2.010600e+06
ridge_stack,370354.479244,0.441470,2.011273e+06
mean_log_blend,403792.274089,0.496109,2.200220e+06


Selected stacker: catboost_stack
Stack OOF: {'mae_original': 359771.8218074995, 'mae_log': 0.4299765361896155, 'tail_mae': 2010600.2392966093}


#### Insights

> Automatic model selection is based on the competition-scale OOF MAE, not log-loss. The shallow stacker is constrained deliberately because only 14,640 rows are available and the base predictions already contain most of the predictive signal.

---

# Diagnostics <a name="9"></a>

---

The final diagnostics compare branch-level OOF performance, quantify expensive-tail errors, and save failure cases for manual inspection. These outputs are more useful for the next modelling iteration than a single aggregate score.

## OOF Scoreboard

All scores below are evaluated on exactly the same rows and fold assignment.

In [22]:
scoreboard = pd.DataFrame({
    'DeBERTa': regression_metrics(train_df['listPrice'].values, deberta_oof_log),
    'TF-IDF Ridge': regression_metrics(train_df['listPrice'].values, tfidf_oof_log),
    'Structured CatBoost': regression_metrics(train_df['listPrice'].values, structured_oof_log),
    'Stack': regression_metrics(train_df['listPrice'].values, stack_oof_log),
}).T.sort_values('mae_original')
scoreboard.to_csv(CFG.ARTIFACT_DIR / 'oof_scoreboard.csv')
display(scoreboard.style.format('{:,.4f}'))

,mae_original,mae_log,tail_mae
Stack,"359,771.8218",0.4300,"2,010,600.2393"
TF-IDF Ridge,"390,010.4812",0.4498,"2,352,386.3758"
Structured CatBoost,"436,768.5731",0.5510,"2,393,345.0777"
DeBERTa,"594,574.7197",0.7169,"2,057,785.5261"


## Residuals and Tail Behaviour

Residual plots are written as separate figures so they remain readable in reports. The decile table shows where absolute dollar error is concentrated.

In [30]:
stack_oof_price = log_to_price(stack_oof_log)
residual = stack_oof_price - train_df['listPrice'].values
abs_error = np.abs(residual)

In [31]:
diagnostic_df = pd.DataFrame({
    'id': train_df['id'].values,
    'fold': train_df['fold'].values,
    'true_price': train_df['listPrice'].values,
    'pred_price': stack_oof_price,
    'residual': residual,
    'abs_error': abs_error,
})

diagnostic_df['price_decile'] = pd.qcut(
    diagnostic_df['true_price'].rank(method='first'), q=10, labels=False
)

decile_error = diagnostic_df.groupby('price_decile').agg(
    listings=('id', 'count'),
    median_price=('true_price', 'median'),
    mae=('abs_error', 'mean'),
    total_absolute_error=('abs_error', 'sum'),
)

decile_error['error_share_pct'] = (
    100
    * decile_error['total_absolute_error']
    / decile_error['total_absolute_error'].sum()
)

display(decile_error)

,listings,median_price,mae,total_absolute_error,error_share_pct
price_decile,,,,,
0,1464,94900.0,1.123350e+05,1.644584e+08,3.122395
1,1464,209000.0,1.205459e+05,1.764791e+08,3.350620
2,1464,299900.0,1.190902e+05,1.743480e+08,3.310158
3,1464,385000.0,1.289189e+05,1.887373e+08,3.583353
4,1464,459000.0,1.294246e+05,1.894776e+08,3.597408
5,1464,549900.0,1.534696e+05,2.246796e+08,4.265750
6,1464,651187.5,1.790944e+05,2.621942e+08,4.977999
7,1464,799900.0,2.344649e+05,3.432566e+08,6.517044
8,1464,1100000.0,4.066495e+05,5.953349e+08,11.302984


In [33]:
diagnostic_df = pd.DataFrame({
    'id': train_df['id'].values,
    'fold': train_df['fold'].values,
    'true_price': train_df['listPrice'].values,
    'pred_price': stack_oof_price,
    'residual': residual,
    'abs_error': abs_error,
})

diagnostic_df['price_decile'] = pd.qcut(
    diagnostic_df['true_price'].rank(method='first'), q=10, labels=False
)

decile_error = diagnostic_df.groupby('price_decile').agg(
    listings=('id', 'count'),
    median_price=('true_price', 'median'),
    mae=('abs_error', 'mean'),
    total_absolute_error=('abs_error', 'sum'),
)

decile_error['error_share_pct'] = (
    100 * decile_error['total_absolute_error'] / decile_error['total_absolute_error'].sum()
)

display(decile_error)

,listings,median_price,mae,total_absolute_error,error_share_pct
price_decile,,,,,
0,1464,94900.0,1.123350e+05,1.644584e+08,3.122395
1,1464,209000.0,1.205459e+05,1.764791e+08,3.350620
2,1464,299900.0,1.190902e+05,1.743480e+08,3.310158
3,1464,385000.0,1.289189e+05,1.887373e+08,3.583353
4,1464,459000.0,1.294246e+05,1.894776e+08,3.597408
5,1464,549900.0,1.534696e+05,2.246796e+08,4.265750
6,1464,651187.5,1.790944e+05,2.621942e+08,4.977999
7,1464,799900.0,2.344649e+05,3.432566e+08,6.517044
8,1464,1100000.0,4.066495e+05,5.953349e+08,11.302984


## Largest OOF Errors

The most expensive mistakes are saved together with branch predictions and source text. Manual review can reveal missing extraction rules, unusual property types, or language patterns that deserve a dedicated feature.

In [34]:
failure_cases = train_df[['id', 'fold', 'listPrice', 'text']].copy()
failure_cases['deberta_price'] = log_to_price(deberta_oof_log)
failure_cases['tfidf_price'] = log_to_price(tfidf_oof_log)
failure_cases['structured_price'] = log_to_price(structured_oof_log)
failure_cases['stack_price'] = stack_oof_price
failure_cases['abs_error'] = np.abs(failure_cases['stack_price'] - failure_cases['listPrice'])
failure_cases = failure_cases.sort_values('abs_error', ascending=False)
failure_cases.head(100).to_csv(CFG.ARTIFACT_DIR / 'top_100_failure_cases.csv', index=False)
display(failure_cases[['id', 'listPrice', 'deberta_price', 'tfidf_price', 'structured_price', 'stack_price', 'abs_error']].head(20))

,id,listPrice,deberta_price,tfidf_price,structured_price,stack_price,abs_error
11599,RE_010053,80000000.0,3.704281e+06,1.593796e+06,7.033427e+06,4.790189e+06,7.520981e+07
12967,RE_014626,54580000.0,1.061294e+06,2.030036e+06,1.858933e+06,4.371111e+06,5.020889e+07
3634,RE_004711,52250000.0,1.006928e+07,2.186528e+06,1.531988e+06,7.033066e+06,4.521693e+07
10230,RE_012644,35000000.0,1.544173e+06,2.887556e+06,2.354112e+06,3.721330e+06,3.127867e+07
4236,RE_017887,40000000.0,8.886110e+06,3.121576e+06,2.382251e+06,9.957237e+06,3.004276e+07
12366,RE_016473,30875000.0,2.246760e+06,6.486316e+05,6.136235e+05,9.782827e+05,2.989672e+07
10509,RE_012504,38500000.0,6.107327e+06,5.318626e+06,3.619690e+06,8.658606e+06,2.984139e+07
7917,RE_002292,37500000.0,8.347727e+06,2.923087e+06,4.613988e+06,8.451234e+06,2.904877e+07
2834,RE_007055,36000000.0,8.347727e+06,2.903716e+06,3.734904e+06,7.139085e+06,2.886091e+07
1358,RE_014075,28000000.0,3.479850e+06,1.522106e+06,2.057953e+06,4.020556e+06,2.397944e+07


#### Insights

> If the top price decile still contributes a disproportionate share of absolute error, the next iteration should target luxury/location/entity signals rather than only tuning average-case hyperparameters. Branch disagreement in the saved failure table is especially useful for deciding which representation is missing information.

---

# Submission <a name="10"></a>

---

The selected stacker's averaged test predictions are converted to the original price scale and inserted into the official sample submission by `id`. Only a non-negative safety bound and a very high overflow cap are applied; no leaderboard-informed calibration is used.

## Build `submission.csv`

The final cell validates IDs, missing values, and prediction range before writing the competition file.

In [35]:
test_prediction = log_to_price(stack_test_log)
prediction_map = dict(zip(test_df['id'], test_prediction))

submission = sample_submission[['id']].copy()
submission['listPrice'] = submission['id'].map(prediction_map)
assert submission['listPrice'].notna().all()
assert np.isfinite(submission['listPrice']).all()
assert (submission['listPrice'] >= 0).all()

submission_path = CFG.OUTPUT_DIR / 'submission.csv'
submission.to_csv(submission_path, index=False)
print('Saved:', submission_path)
display(submission.head())
display(submission['listPrice'].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_frame())

Saved: /workspace/outputs/nb1_deberta_late_fusion/submission.csv


,id,listPrice
0,RE_015637,518871.998679
1,RE_005176,342176.249399
2,RE_013544,879175.000106
3,RE_016667,338958.364649
4,RE_011479,933702.958210


,listPrice
count,3.659000e+03
mean,7.613037e+05
std,1.117154e+06
min,1.779033e+04
50%,5.098175e+05
90%,1.345306e+06
95%,2.132054e+06
99%,5.275997e+06
max,1.549413e+07


## Saved Artifacts

The notebook writes the DAPT checkpoint, fold-level DeBERTa models, OOF/test predictions for every branch, TF-IDF/Ridge artifacts, CatBoost models, stacker artifacts, diagnostics, and the final submission under `outputs/nb1_deberta_late_fusion/` (or `/kaggle/working/...` on Kaggle).

In [36]:
artifact_inventory = sorted(
    [str(path.relative_to(CFG.OUTPUT_DIR)) for path in CFG.OUTPUT_DIR.rglob('*') if path.is_file()]
)
print(f'{len(artifact_inventory)} files saved under {CFG.OUTPUT_DIR}')
for item in artifact_inventory[:80]:
    print(item)
if len(artifact_inventory) > 80:
    print(f'... and {len(artifact_inventory) - 80} more files')

141 files saved under /workspace/outputs/nb1_deberta_late_fusion
artifacts/deberta_fold_metrics.csv
artifacts/deberta_folds/fold_0/best_model/added_tokens.json
artifacts/deberta_folds/fold_0/best_model/config.json
artifacts/deberta_folds/fold_0/best_model/model.safetensors
artifacts/deberta_folds/fold_0/best_model/special_tokens_map.json
artifacts/deberta_folds/fold_0/best_model/spm.model
artifacts/deberta_folds/fold_0/best_model/tokenizer.json
artifacts/deberta_folds/fold_0/best_model/tokenizer_config.json
artifacts/deberta_folds/fold_0/best_model/training_args.bin
artifacts/deberta_folds/fold_0/checkpoints/checkpoint-183/added_tokens.json
artifacts/deberta_folds/fold_0/checkpoints/checkpoint-183/config.json
artifacts/deberta_folds/fold_0/checkpoints/checkpoint-183/model.safetensors
artifacts/deberta_folds/fold_0/checkpoints/checkpoint-183/optimizer.pt
artifacts/deberta_folds/fold_0/checkpoints/checkpoint-183/rng_state.pth
artifacts/deberta_folds/fold_0/checkpoints/checkpoint-183/sche

#### Final Notes

> For a first Vast.ai run, keep all defaults and run top-to-bottom. For rapid iteration after artifacts exist, disable expensive branches individually and let the notebook reload their saved OOF/test predictions.

> The most important comparison is not Transformer versus TF-IDF in isolation, but whether their OOF residuals are complementary. The stacker only helps when the branches make meaningfully different mistakes.